# MCDI500 — Programación para la Ciencia de Datos
## Proyecto Transversal — Fase 3: Núcleo Algorítmico y Programación Orientada a Objetos

**Título del proyecto:** Factores socioeconómicos y de preparación previa asociados al rendimiento académico

| Campo | Detalle |
|---|---|
| **Curso** | MCDI500 — Programación para la Ciencia de Datos |
| **Docente** | Omar Salinas Silva |
| **Grupo** | Grupo 7 |
| **Integrantes** | Juan de Dios Díaz Ríos · Francisco Fariña Molina · Constanza Moreno Giacometto · Yenne Sepúlveda Jerez |
| **Fecha** | Junio 2026 |
| **Dataset** | Student Performance Dataset (Cortez & Silva, 2008) — UCI Machine Learning Repository |

---

## Índice

1. [Contexto: de la Fase 2 a la Fase 3](#1)
2. [Diseño de la clase `PreprocesadorAsignatura`](#2)
3. [Uso de la clase con datos reales](#3)
4. [Herencia y polimorfismo: `PreprocesadorMatematicas` y `PreprocesadorPortugues`](#4)
5. [Recursividad](#5)
   - 5.1 Esqueleto general de un algoritmo recursivo
   - 5.2 Merge Sort aplicado a las notas del dataset
6. [Eficiencia —  Comparativa de tiempos de ejecución](#6)
7. [Validación y exportación final mediante objetos](#7)
8. [Bibliografia (APA 7) ](#8)


---
<a id='1'></a>
## 1. Contexto: de la Fase 2 a la Fase 3

En la **Fase 2** el equipo desarrolló un pipeline de **obtención, limpieza y transformación de datos**
implementado mediante **funciones** encapsuladas en `src/functions.py`: `cargar_dataset()`,
`resumen_dataset()`, `validar_rangos()`, `detectar_outliers_iqr()`, `analizar_correlaciones()`,
`limpiar_dataset()`, `winsorizacion()`, `codificar_binarias()`, `codificar_ohe()`,
`crear_variables_derivadas()`, `validar_dataset_final()` y `exportar_dataset()`.

Ese pipeline funciona correctamente, pero los datos (`df_mat`, `df_por`, `df_mat_enc`, `df_por_enc`,
etc.) y los pasos del proceso quedan **sueltos** en el notebook: cada función recibe un `DataFrame`,
lo transforma y lo retorna, y es responsabilidad de quien lee el notebook recordar en qué estado
quedó cada variable.

La **Fase 3** consolida ese pipeline dentro de un **núcleo algorítmico orientado a objetos**. La idea
central es la misma que vimos con la clase `Preprocesador` en la guía de la Semana 2:
juntar en un mismo lugar — un **objeto** — **los datos** (`self.df`, `self.df_enc`) y **las acciones**
que operan sobre ellos (`cargar()`, `explorar()`, `limpiar()`, `transformar()`, `validar()`,
`exportar()`).

**Lo importante:** esta capa de POO **no reemplaza** las funciones de la Fase 2 — las **reutiliza**.
Cada método de la nueva clase `PreprocesadorAsignatura` (definida en `src/clases.py`) llama
internamente a las funciones ya validadas de `functions.py`. Esto cumple con el objetivo de la
Fase 3: *"integrar estos resultados previos con la implementación inicial del núcleo algorítmico del
sistema, incorporando principios de POO"*.

Adicionalmente, en este notebook se incorporan:
- **Herencia y polimorfismo** (Sección 4): `PreprocesadorMatematicas` y `PreprocesadorPortugues`
  heredan de `PreprocesadorAsignatura` y cada una implementa su propia interpretación de los
  resultados de correlación obtenidos en la Fase 2.
- **Recursividad** (Sección 5): un ejemplo aplicado al proyecto — ordenando los notas de g3 mediante logica recursiva, separandolo la data en 2
- **Eficiencia y complejidad** (Sección 6): comparación de tiempos de ejecución entre
  implementaciones con bucles y sus equivalentes vectorizados con NumPy/Pandas, en relación a la aprobación (g3>=10)

In [ ]:
# ============================================================
# ENTORNO E IMPORTACIÓN DE MÓDULOS DEL PROYECTO
# ============================================================
import sys, os
sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import timeit

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='muted')

# Funciones de la Fase 2 (pipeline funcional ya validado)
from functions import (
    cargar_dataset, mostrar_primeras_filas, resumen_dataset,
    validar_rangos, detectar_outliers_iqr, analizar_correlaciones,
    analizar_g3_cero, limpiar_dataset, winsorizacion,
    codificar_binarias, codificar_ohe, crear_variables_derivadas,
    validar_dataset_final, exportar_dataset, integrar_datasets, merge_sort, 
    combinar,aprobado_bucle, aprobado_vectorizado
)

# Núcleo algorítmico de la Fase 3: clases (POO) y funciones recursivas
from clases import PreprocesadorAsignatura, PreprocesadorMatematicas, PreprocesadorPortugues


print("✓ Entorno listo. Módulos de Fase 2 (functions) y Fase 3 (clases) importados.")